# Optical control: same forgeries on Sentinel-2 (#47)

Does the copy-move model rely on SAR (speckle, radar texture), or only on
duplication? This notebook makes the **same forgeries on optical images**
(Sentinel-2, over the two SAR test scenes) so that the evaluation notebook
can score the SAR model on them **as it is**: no retraining, settings
frozen on its SAR val. Design and reading of the result: `docs/report.md`,
section 5.3.

Two data sources are made, each at levels 1-3:
- `_optical`: grey = mean of B02 / B03 / B04, given the distribution of the
  SAR training patches in dB;
- `_optical_speckle`: the same with synthetic speckle (ENL 4.4), the control.

Then, in `features/evaluation/evaluation.ipynb`: `TRAINED_ON_LEVEL = 3`,
`DATASET_LEVEL = 1, 2 or 3`, `DATA_SOURCE = "_optical"` or
`"_optical_speckle"`. **Runtime: CPU** (no GPU needed here); high RAM helps
(a Sentinel-2 band is 10980 x 10980 px).

| Cell | What it does |
|---|---|
| 1 | Parameters (your values) |
| 2 | Mount Drive, get the code, install the packages |
| 3 | Copy the 3 bands of each Sentinel-2 product from Drive to the VM |
| 4 | SAR reference distribution, conversion, tiling, forgeries (both sources, levels 1-3) |
| 5 | Zip the datasets onto Drive, with `conversion.json` |
| 6 | Look at one patch: optical, optical + speckle, and its level-3 forgery (saved in `data/figures/` on Drive) |

In [ ]:
REPO_URL = "https://github.com/bonlieuguillaume/aletheia.git"
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/aletheia"

# Sentinel-2 L2A products, searched by name in SCENES_S2_DIR or one folder below.
# Key = the SAR TEST scene each one covers (docs/v0.1_south_american_forest.csv),
# so that the per-scene results line up with SAR.
SCENES_S2_DIR = DRIVE_ROOT + "/data/scenes_S2"
SCENES = {
    "ds_forest_1": "S2A_MSIL2A_20220513T143731_N0510_R096_T19LHK_20241128T052531.SAFE",
    "ds_forest_9": "S2A_MSIL2A_20181012T141041_N0500_R110_T21NZE_20230730T000834.SAFE",
}
# SAR dataset of the reference model: its authentic training patches give the
# distribution (in dB) the optical grey levels are mapped onto.
SAR_DATASET_ZIP = DRIVE_ROOT + "/data/dataset_v0.1_level3.zip"
METHOD = "histogram"        # "histogram" (whole distribution) or "affine" (mean / std only)
LEVELS = (1, 2, 3)
SEED = 0                    # same seed as the SAR datasets: same forgery generators

WORK_DIR = "/content/optical"                        # VM disk
OUT_DIR = DRIVE_ROOT + "/data"                       # dataset_v0.1_level<L>_optical[_speckle].zip

In [ ]:
# Mount Google Drive, get the latest code from GitHub, install the packages.
# Everything happens on the Colab VM: nothing is installed on your PC.
import os
import subprocess
import sys

from google.colab import drive
drive.mount("/content/drive")

REPO_DIR = "/content/aletheia"
if os.path.isdir(REPO_DIR + "/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run(["pip", "install", "-q", "-r", REPO_DIR + "/requirements.txt"], check=True)

os.chdir(REPO_DIR + "/features/optical")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("Code at commit", COMMIT)

In [ ]:
# Only the three 10 m visible bands are needed: a few hundred MB per product,
# instead of the whole .SAFE (Drive is slow per file).
import shutil
from pathlib import Path

from optical import find_s2_bands

local_safe_dirs = {}
for scene, safe_name in SCENES.items():
    # At most one folder deep (scenes_S2/<product> or scenes_S2/<scene>/<product>):
    # a recursive search would walk the thousands of files of every .SAFE on Drive.
    found = sorted(Path(SCENES_S2_DIR).glob(safe_name)) + sorted(Path(SCENES_S2_DIR).glob("*/" + safe_name))
    if len(found) != 1:
        raise FileNotFoundError(f"{safe_name}: found {len(found)} times under {SCENES_S2_DIR}")
    local_dir = Path(WORK_DIR) / "s2" / scene
    local_dir.mkdir(parents=True, exist_ok=True)
    for band, path in find_s2_bands(found[0]).items():
        if not (local_dir / path.name).exists():
            shutil.copy(path, local_dir / path.name)
        print(f"{scene}: {band} <- {path.name}")
    local_safe_dirs[scene] = local_dir

In [ ]:
from optical import build_optical_datasets, sar_reference

reference = sar_reference(SAR_DATASET_ZIP)
print(f"SAR reference ({reference['n_patches']} training patches of {reference['source']}): "
      f"mean {reference['mean']:.2f} dB, std {reference['std']:.2f} dB")

datasets = build_optical_datasets(local_safe_dirs, reference, WORK_DIR, levels=LEVELS,
                                  method=METHOD, seed=SEED)

In [ ]:
import json

from features.sar_io.sar_io import zip_folder

# What the datasets came from, next to them on Drive (reproducibility).
conversion = {
    "code_commit": COMMIT,
    "scenes": SCENES,
    "method": METHOD,
    "seed": SEED,
    "levels": list(LEVELS),
    "sar_reference": {k: v for k, v in reference.items() if k != "quantiles"},
}
for (variant, level), dataset_dir in datasets.items():
    (dataset_dir / "conversion.json").write_text(json.dumps({**conversion, "variant": variant}, indent=2))
    zip_folder(dataset_dir, f"{OUT_DIR}/dataset_v0.1_level{level}{variant}.zip")

In [ ]:
import os

import matplotlib.pyplot as plt

from features.copy_move.copy_move import show_forgery
from features.evaluation.evaluation import to_display
from features.sar_io.sar_io import read_manifest, read_tif

# One patch seen three ways: optical, optical + speckle, and its level-3
# forgery (the patch ids are the same in both sources: same tiling).
FIGURES_DIR = DRIVE_ROOT + "/data/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)
level = max(LEVELS)
manifest = read_manifest(datasets[("_optical", level)] / "manifest.csv")
patch_id = manifest[manifest["label"] == 1].sample(1, random_state=SEED)["patch_id"].iloc[0]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, variant in zip(axes, ["_optical", "_optical_speckle"]):
    patch = read_tif(datasets[(variant, level)] / "real" / f"{patch_id}.tif")
    ax.imshow(to_display(patch), cmap="gray")
    ax.set_title(f"{patch_id} {variant}")
    ax.axis("off")
fig.savefig(f"{FIGURES_DIR}/optical_sources_{patch_id}.png", dpi=150, bbox_inches="tight")
plt.show()

for variant in ["_optical", "_optical_speckle"]:
    fig = show_forgery(datasets[(variant, level)], patch_id)
    fig.savefig(f"{FIGURES_DIR}/forgeries_level{level}{variant}_{patch_id}.png", dpi=150, bbox_inches="tight")
    plt.show()